## I. Các gói thư viện sử dụng

In [1]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="huggingface_hub.*")

In [18]:
!pip install python-dotenv huggingface_hub datasets -q

In [19]:
import os
import re
import unicodedata

import numpy as np
import pandas as pd
from datasets import load_dataset, Dataset, DatasetDict
from dotenv import find_dotenv, load_dotenv
from huggingface_hub import login
from sklearn.model_selection import train_test_split

## II. Thiết lập môi trường làm việc
1. Kết nối google drive
2. Nạp biến môi trường từ .env
3. Kết nối Hugging Face

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
root_path = "/content/drive/MyDrive/BTL_AI_13_IT2302/AIWritingIndicator-13/SourceCode/"
%cd {root_path}
!pwd

/content/drive/.shortcut-targets-by-id/17JBaRqvfDFY0j8cN0vwu8kzqwtDISdDR/BTL_AI_13_IT2302/AIWritingIndicator-13/SourceCode
/content/drive/.shortcut-targets-by-id/17JBaRqvfDFY0j8cN0vwu8kzqwtDISdDR/BTL_AI_13_IT2302/AIWritingIndicator-13/SourceCode


In [6]:
env_path = find_dotenv()

if env_path:
    load_dotenv(env_path)
else:
    print("Không tìm thấy file .env!")

In [7]:
hf_token = os.getenv("HF_TOKEN")

if hf_token:
    login()
else:
    print("Không tồn tại env HF_TOKEN!")

## III. Tiền xử lý dữ liệu

In [16]:
class AIDataPreprocessor:
    def __init__(self, input_path, output_path, hf_repo_id=None):
        self.input_path = input_path
        self.output_path = output_path
        self.hf_repo_id = hf_repo_id
        self.df = None

    def load_data(self):
        self.df = pd.read_csv(self.input_path)

    @staticmethod
    def clean_and_normalize_text(text):
        text = str(text)
        text = unicodedata.normalize('NFC', text)
        text = text.lower()
        text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
        text = re.sub(r'<.*?>', '', text)
        text = re.sub(r'[^\w\s]', ' ', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    def clean_and_handle_missing(self):
        self.df = self.df.dropna(subset=['Text', 'segmented_text', 'Label'])
        self.df = self.df[self.df['Text'].str.strip() != '']
        self.df = self.df[self.df['segmented_text'].str.strip() != '']
        self.df['segmented_text'] = self.df['segmented_text'].apply(self.clean_and_normalize_text)

        numeric_cols = self.df.select_dtypes(include=['float64', 'int64']).columns
        self.df[numeric_cols] = self.df[numeric_cols].apply(lambda col: col.fillna(col.mean()))
        self.df = self.df.drop_duplicates(subset=['Text'], keep='first')

    def select_features(self):
        if 'number_count' in self.df.columns and 'word_count' in self.df.columns:
            self.df['number_ratio'] = (self.df['number_count'] / self.df['word_count']).fillna(0)

        columns_to_drop = [
            'language', 'is_valid_vi', 'stopwords_found', 'char_count',
            'email_count', 'missing_punctuation', 'noun_ratio', 'verb_ratio',
            'capitalized_words', 'number_count', 'long_words_count', 'perplexity','adverb_ratio', 'word_count'
        ]
        self.df = self.df.drop(columns=[col for col in columns_to_drop if col in self.df.columns])

    def export_and_push_to_hf(self):
        self.df.to_csv(self.output_path, index=False)

        if self.hf_repo_id:
            try:
                train_df, testval_df = train_test_split(
                    self.df, test_size=0.3, stratify=self.df['Label'], random_state=42
                )
                val_df, test_df = train_test_split(
                    testval_df, test_size=1/3, stratify=testval_df['Label'], random_state=42
                )

                final_dataset = DatasetDict({
                    'train': Dataset.from_pandas(train_df, preserve_index=False),
                    'validation': Dataset.from_pandas(val_df, preserve_index=False),
                    'test': Dataset.from_pandas(test_df, preserve_index=False)
                })

                final_dataset.push_to_hub(self.hf_repo_id)
                print("Tải lên Hugging Face thành công!")
            except Exception as e:
                print(f"Có lỗi xảy ra khi đẩy lên Hugging Face: {e}")

    def run_pipeline(self):
        self.load_data()
        self.clean_and_handle_missing()
        self.select_features()
        self.export_and_push_to_hf()
        return self.df

In [17]:
INPUT_FILE = "Data/vietnamese_news_human_ai_eda.csv"
OUTPUT_FILE = "Data/vietnamese_news_human_ai_cleaned.csv"
HF_REPO = "JuniorThanh/vietnamese_news_human_ai_cleaned"

preprocessor = AIDataPreprocessor(
    input_path=INPUT_FILE,
    output_path=OUTPUT_FILE,
    hf_repo_id=HF_REPO
)

cleaned_df = preprocessor.run_pipeline()

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/7 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :  89%|########8 | 15.1MB / 17.0MB            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########| 2.43MB / 2.43MB            

README.md: 0.00B [00:00, ?B/s]

Tải lên Hugging Face thành công!
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Text                10000 non-null  object 
 1   Label               10000 non-null  int64  
 2   sentence_count      10000 non-null  int64  
 3   ttr                 10000 non-null  float64
 4   special_char_ratio  10000 non-null  float64
 5   caps_ratio          10000 non-null  float64
 6   segmented_text      10000 non-null  object 
 7   stopwords_ratio     10000 non-null  float64
 8   entropy             10000 non-null  float64
 9   adj_ratio           10000 non-null  float64
 10  pronoun_ratio       10000 non-null  float64
 11  conjunction_ratio   10000 non-null  float64
 12  burstiness          10000 non-null  float64
 13  number_ratio        10000 non-null  float64
dtypes: float64(10), int64(2), object(2)
memory usage: 1.1+ MB


In [ ]:
cleaned_df.info()